## Imports

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

## Load LinkedIn postings

In [5]:
postings=pd.read_csv('postings.csv',
    usecols=['job_id', 'company_name', 'title', 'description', 'formatted_experience_level','skills_desc', 'location', 'formatted_work_type'],
    low_memory=False)

print(postings.shape)
postings.head()

(123849, 8)


,job_id,company_name,title,description,location,formatted_work_type,formatted_experience_level,skills_desc
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,"Princeton, NJ",Full-time,NaN,Requirements: \n\nWe are seeking a College or ...
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...","Fort Collins, CO",Full-time,NaN,NaN
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,"Cincinnati, OH",Full-time,NaN,We are currently accepting resumes for FOH - A...
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,"New Hyde Park, NY",Full-time,NaN,This position requires a baseline understandin...
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,"Burlington, IA",Full-time,NaN,NaN


In [4]:
# CELL — Peek at just the header row, cheap and fast even on a 500MB file
header = pd.read_csv('postings.csv', nrows=0)
print(header.columns.tolist())

['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


## Drop rows with no usable description text

In [6]:
postings = postings.dropna(subset=['title', 'description'])
print("After dropping missing title/description:", postings.shape)

After dropping missing title/description: (123842, 8)


## Map 24 Resume.csv categories to relevant LinkedIn job title keywords

In [7]:

category_keywords = {
    'ACCOUNTANT': ['accountant', 'accounting'],
    'ADVOCATE': ['lawyer', 'attorney', 'legal counsel', 'advocate'],
    'AGRICULTURE': ['agriculture', 'farm', 'agronomist'],
    'APPAREL': ['apparel', 'fashion', 'garment'],
    'ARTS': ['artist', 'graphic design', 'illustrator'],
    'AUTOMOBILE': ['automotive', 'auto mechanic', 'vehicle'],
    'AVIATION': ['pilot', 'aviation', 'aircraft'],
    'BANKING': ['banker', 'banking', 'bank teller'],
    'BPO': ['call center', 'bpo', 'customer service representative'],
    'BUSINESS-DEVELOPMENT': ['business development'],
    'CHEF': ['chef', 'cook', 'culinary'],
    'CONSTRUCTION': ['construction', 'site manager', 'foreman'],
    'CONSULTANT': ['consultant', 'consulting'],
    'DESIGNER': ['designer', 'ux designer', 'ui designer'],
    'DIGITAL-MEDIA': ['digital media', 'social media', 'content creator'],
    'ENGINEERING': ['engineer', 'engineering'],
    'FINANCE': ['finance', 'financial analyst'],
    'FITNESS': ['fitness', 'personal trainer', 'gym'],
    'HEALTHCARE': ['nurse', 'healthcare', 'medical', 'physician'],
    'HR': ['human resources', 'hr generalist', 'recruiter'],
    'INFORMATION-TECHNOLOGY': ['software engineer', 'it support', 'developer', 'programmer'],
    'PUBLIC-RELATIONS': ['public relations', 'pr specialist', 'communications'],
    'SALES': ['sales representative', 'sales associate', 'account executive'],
    'TEACHER': ['teacher', 'instructor', 'educator'],
}

print(f"{len(category_keywords)} categories mapped")

24 categories mapped


## Filter postings: keep only rows whose title matches at least one keyword for some category

In [8]:
def match_category(title,keyword_map):
    title_lower=str(title).lower()
    for category, keywords in keyword_map.items():
        for kw in keywords:
            if kw in title_lower:
                return category
    return None

postings['matched_category'] = postings['title'].apply(lambda x: match_category(x, category_keywords))
filtered_postings =postings[postings['matched_category'].notna()].copy()

print(f"Filtered from {len(postings)} down to {len(filtered_postings)} postings")
print(filtered_postings['matched_category'].value_counts())

Filtered from 123842 down to 39330 postings
matched_category
ENGINEERING               10875
HEALTHCARE                 7693
SALES                      3485
INFORMATION-TECHNOLOGY     2725
ACCOUNTANT                 2108
CONSULTANT                 1956
TEACHER                    1111
CONSTRUCTION               1089
FINANCE                     931
HR                          900
BPO                         881
ADVOCATE                    832
BUSINESS-DEVELOPMENT        790
DESIGNER                    769
BANKING                     726
CHEF                        633
PUBLIC-RELATIONS            476
AUTOMOBILE                  444
DIGITAL-MEDIA               254
ARTS                        205
APPAREL                     148
AVIATION                    142
FITNESS                      83
AGRICULTURE                  74
Name: count, dtype: int64


In [11]:
# Spot-check the mapping quality
for cat in ['ENGINEERING', 'HEALTHCARE', 'CONSULTANT', 'SALES', 'BPO']:
    print(f"\n=== {cat} ===")
    print(filtered_postings[filtered_postings['matched_category']==cat]['title'].sample(10, random_state=1).tolist())


=== ENGINEERING ===
['Engineering Department Intern - NYC Ferry', 'Project Engineer', 'Senior Front-End Engineer (remote)', 'Senior Manufacturing Engineer', 'Civil Engineer', 'Senior Machine Learning Engineer (Python, PySpark, SQL)', 'Torchlight Real-Time Engineer', 'Founding Engineer', 'FPGA Development Engineer', 'Lead QA Engineer']

=== HEALTHCARE ===
['CVICU Registered Nurse', 'Registered Nurse - 2 C Observation Overflow', 'Medical Assistant', 'Registered Nurse RN', 'Registered Nurse', 'Healthcare Recruiter', 'Sell Healthcare NO Licensing Required', 'Registered Nurse (RN) Neuro Telemetry - Per Diem', 'Family Practice-Without OB Physician - $300,000/yearly - $400,000/yearly', 'Registered Nurse']

=== CONSULTANT ===
['Xfinity Retail Sales Consultant, Full Time (Hermitage)', 'Manager, Risk Advisory and Consulting (Financial)', 'Sr Director - Consulting & Analytics - Nationals segment', 'Required Data Consultant - USC GC ONLY - Local to NY', 'Implementation Consultant - CPA', 'Princip

In [12]:
# Spot-check all 24 categories at once
for cat in sorted(filtered_postings['matched_category'].unique()):
    titles = filtered_postings[filtered_postings['matched_category']==cat]['title'].sample(
        min(10, len(filtered_postings[filtered_postings['matched_category']==cat])), random_state=1
    ).tolist()
    print(f"\n=== {cat} (n={len(filtered_postings[filtered_postings['matched_category']==cat])}) ===")
    for t in titles:
        print(f"  {t}")


=== ACCOUNTANT (n=2108) ===
  Community Accountant
  Accounting Manager - Gainesville FL
  Staff Accountant
  Accounting Advisory (Full-time)
  Senior Director, Corporate Accounting and Global Consolidations (Remote)
  Cost Accountant II
  Accountant - Manufacturing - Plant
  Accountant (140859)
  Senior Accountant
  Project Accountant

=== ADVOCATE (n=832) ===
  Associate Attorney
  Associate Attorney
  Patient Care Advocate
  Insurance Attorney
  Probate Litigation Attorney in Arlington, TX
  Associate Attorney - Commercial Litigation
  Senior Legal Counsel, International Programs (One-Year Fellowship)
  Associate Attorney
  Customer Advocate
  Legal Counsel Region NA

=== AGRICULTURE (n=74) ===
  Farm Bureau Agent
  Supply Chain Assistant (Agriculture Chemicals)
  Associate Recruiter - Agriculture
  Herdsperson FARM Trainee - Farm 76921
  Feed Mill Manager - Poultry Farm
  Agriculture Equipment Sales
  Lifestyle Director - Barney Farms
  Farm Hand/Animal Care Associate
  State Farm

In [13]:
exclude_terms = {
    'CONSULTANT': ['sales consultant', 'retail consultant'],
    'HEALTHCARE': ['recruiter', 'sell healthcare'],
    'AGRICULTURE': ['farm bureau', 'state farm', 'equipment sales', 'lifestyle director'],
    'ADVOCATE': ['patient care advocate', 'customer advocate'],
    'AVIATION': ['pilot plant'],
    'PUBLIC-RELATIONS': ['it communications', 'parole communications', 'pbx operator'],
}

def match_category_v2(title, keyword_map, exclude_map=None):
    title_lower = str(title).lower()
    for category, keywords in keyword_map.items():
        excludes = exclude_map.get(category, []) if exclude_map else []
        if any(ex in title_lower for ex in excludes):
            continue
        for kw in keywords:
            if kw in title_lower:
                return category
    return None

postings['matched_category'] = postings['title'].apply(
    lambda t: match_category_v2(t, category_keywords, exclude_terms)
)
filtered_postings = postings[postings['matched_category'].notna()].copy()
print(filtered_postings['matched_category'].value_counts())

matched_category
ENGINEERING               10875
HEALTHCARE                 7675
SALES                      3486
INFORMATION-TECHNOLOGY     2725
ACCOUNTANT                 2108
CONSULTANT                 1688
TEACHER                    1111
CONSTRUCTION               1089
FINANCE                     931
HR                          921
BPO                         881
ADVOCATE                    815
BUSINESS-DEVELOPMENT        790
DESIGNER                    769
BANKING                     726
CHEF                        633
PUBLIC-RELATIONS            470
AUTOMOBILE                  444
DIGITAL-MEDIA               254
ARTS                        205
APPAREL                     148
AVIATION                    134
FITNESS                      83
AGRICULTURE                  64
Name: count, dtype: int64


## Cap per-category count

In [15]:
CAP_PER_CATEGORY=100

filtered_postings_capped=(filtered_postings.groupby('matched_category',group_keys=False).apply(lambda x: x.sample(min(len(x), CAP_PER_CATEGORY), random_state=42)).reset_index(drop=True))

print("After capping to", CAP_PER_CATEGORY, "per category:", filtered_postings_capped.shape)
print(filtered_postings_capped['matched_category'].value_counts())

After capping to 100 per category: (2347, 9)
matched_category
ACCOUNTANT                100
ADVOCATE                  100
APPAREL                   100
ARTS                      100
AVIATION                  100
AUTOMOBILE                100
BANKING                   100
BPO                       100
FINANCE                   100
BUSINESS-DEVELOPMENT      100
CHEF                      100
CONSTRUCTION              100
CONSULTANT                100
DESIGNER                  100
DIGITAL-MEDIA             100
ENGINEERING               100
SALES                     100
HEALTHCARE                100
HR                        100
INFORMATION-TECHNOLOGY    100
TEACHER                   100
PUBLIC-RELATIONS          100
FITNESS                    83
AGRICULTURE                64
Name: count, dtype: int64


C:\Users\HP\AppData\Local\Temp\ipykernel_23260\417886802.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  filtered_postings_capped=(filtered_postings.groupby('matched_category',group_keys=False).apply(lambda x: x.sample(min(len(x), CAP_PER_CATEGORY), random_state=42)).reset_index(drop=True))


## Clean the JD text

In [16]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

filtered_postings_capped['jd_clean']=filtered_postings_capped['description'].apply(clean_text)
filtered_postings_capped.head

<bound method NDFrame.head of           job_id                     company_name  \
0     3903845778                     Michael Page   
1     3906248938              GXO Logistics, Inc.   
2     3902742846                  Cohen & Company   
3     3905851456              OmniForce Solutions   
4     3891079198    Virginia Department of Health   
...          ...                              ...   
2342  3895210847                              NaN   
2343  3888408574                         CodePath   
2344  3905887538           Meeting Street Schools   
2345  3901944731  East Aurora School District 131   
2346  3895527284                 Uncommon Schools   

                                                  title  \
0             Senior Accounting Manager - Manufacturing   
1                                        Sr. Accountant   
2     Assurance Staff Accountant - October 2024 or J...   
3                                            Accountant   
4                    Accounts Receivab

## Save filtered set 

In [17]:
filtered_postings_capped.to_csv('filtered_linkedin_postings.csv', index=False)
print(f"Saved {len(filtered_postings_capped)} filtered postings")

Saved 2347 filtered postings
